# 01 – Audit data (CP1, FR-01)

Angka dihitung dari **gambar lokal**, lalu dibandingkan dengan Tabel 3.2 proposal (150 jinak / 60 ganas pasien; 7.657 / 3.487 citra). Selisih dilaporkan apa adanya. Dataset hanya dibaca.

**Notebook mandiri (Versi 2):** tanpa folder `src`, tanpa Docker. Semua kode ada di sel-sel notebook ini. Kode ditulis dari Versi 1 dan **belum dijalankan** di mesin penulis (tanpa torch/GPU); jalankan dari atas ke bawah di mesin RTX 3070 Ti. Hasil yang tidak dapat dihitung dilaporkan apa adanya, bukan diisi.

## 1. Pustaka tambahan (opsional)

In [ ]:
INSTALL = False      # True hanya sekali, di venv yang sama dengan notebook 00 (torch CUDA sudah terpasang di sana)

if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "pillow", "openpyxl", "pyyaml"])

## 2. Impor, konfigurasi, utilitas, dan pemeriksaan checkpoint

In [ ]:
from __future__ import annotations

import contextlib, copy, datetime as dt, gc, hashlib, importlib, io, json, math, os, platform, random, re, shutil, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import yaml
from IPython.display import display, Image

# expandable_segments mengurangi fragmentasi memori, tetapi belum didukung di Windows.
if platform.system() != "Windows":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn
import torch.nn.functional as F


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "configs" / "base.yaml").exists():
            return q
    raise FileNotFoundError("configs/base.yaml tidak ditemukan dari direktori kerja ke atas")


ROOT = find_root()
os.chdir(ROOT)

_ENV = {"LARYNX_DATASET": "dataset", "LARYNX_OUTPUTS": "outputs", "LARYNX_HF_CACHE": "hf_cache"}


def deep_merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out


def _load(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = cfg.pop("_base", None)
    return deep_merge(_load(path.parent / base), cfg) if base else cfg


def load_merged(path) -> dict:
    cfg = _load(Path(path))
    for env, key in _ENV.items():
        if os.environ.get(env):
            cfg["paths"][key] = os.environ[env]
    for key, val in cfg["paths"].items():           # jalur relatif -> mutlak terhadap folder Versi 2
        if not Path(val).is_absolute():
            cfg["paths"][key] = str((ROOT / val).resolve())
    return cfg


def P(cfg: dict, *parts: str) -> Path:
    return Path(cfg["paths"]["outputs"]).joinpath(*parts)


def config_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()


def job_seed(base_seed: int, *parts) -> int:
    key = "|".join([str(base_seed), *map(str, parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def atomic_write_bytes(path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def atomic_write_json(path, obj) -> None:
    atomic_write_bytes(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8"))


def atomic_torch_save(obj, path) -> None:
    buf = io.BytesIO(); torch.save(obj, buf); atomic_write_bytes(path, buf.getvalue())


def is_done(job_dir, cfg_hash: str) -> bool:
    done = Path(job_dir) / "DONE"
    return done.exists() and done.read_text(encoding="utf-8").strip() == cfg_hash


def mark_done(job_dir, cfg_hash: str) -> None:
    atomic_write_bytes(Path(job_dir) / "DONE", cfg_hash.encode("utf-8"))


def git_commit() -> str:
    # Versi 2 tanpa Docker: hash commit dari git (bila ada), ditambah -dirty bila ada perubahan belum di-commit.
    try:
        c = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=15).stdout.strip()
        d = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"], capture_output=True, text=True, timeout=15).stdout.strip()
        return (c or "tanpa-commit") + ("-dirty" if d and c else "")
    except Exception:  # noqa: BLE001
        return "tanpa-commit"


GIT_COMMIT = git_commit()
CFG_PATH = "configs/base.yaml"
cfg = load_merged(CFG_PATH)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = P(cfg, "cp0")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT   :", ROOT)
print("dataset:", cfg["paths"]["dataset"])
print("outputs:", cfg["paths"]["outputs"])
print("device :", DEV, "| commit:", GIT_COMMIT, "| config_hash:", config_hash(cfg)[:12])

def free_gpu() -> None:
    # Pengganti 'proses baru per job' di Versi 1: dipanggil di akhir tiap job agar memori GPU kembali bersih.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


WORK_DIR = Path(cfg["paths"]["outputs"]) / "_workers"


def install_module(name: str, src: str):
    # Windows memakai 'spawn': fungsi/kelas yang dipakai proses anak harus berada di modul yang dapat diimpor. Modul dibuat
    # saat notebook berjalan (satu-satunya sumber kodenya ada di sel notebook ini) lalu diimpor, termasuk oleh proses anak.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p = WORK_DIR / f"{name}.py"
    changed = (not p.exists()) or p.read_text(encoding="utf-8") != src
    if changed:
        p.write_text(src, encoding="utf-8")
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod) if changed and name in sys.modules else mod

In [ ]:
# ---------------------------------------------------------------- checkpoint, persetujuan manusia, LOG.md
# Pengganti checkpoint.py dan approve.py Versi 1. Persetujuan MANUSIA tidak pernah dipanggil otomatis oleh notebook.
LULUS, GAGAL, MENUNGGU, TIDAK_TERSEDIA = "LULUS", "GAGAL", "MENUNGGU", "TIDAK_TERSEDIA"
DEC_PATH, BASE_PATH, LOG_PATH = ROOT / "configs" / "decisions.yaml", ROOT / "configs" / "base.yaml", ROOT / "LOG.md"


def file_sha256(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_manifest(out_dir, files, extra=None) -> Path:
    out_dir = Path(out_dir)
    entries = {}
    for f in files:
        p = Path(f)
        if not p.exists():
            raise FileNotFoundError(f"Keluaran checkpoint tidak ditemukan: {p}")
        key = str(p.relative_to(out_dir)) if p.is_relative_to(out_dir) else str(p)
        entries[key] = {"sha256": file_sha256(p), "bytes": p.stat().st_size}
    mp = out_dir / "manifest.json"
    atomic_write_json(mp, {"files": entries, **(extra or {})})
    return mp


def _item(nama: str, status: str, detail: str) -> dict:
    return {"item": nama, "status": status, "detail": detail}


def _review_item(out_dir) -> dict:
    out_dir = Path(out_dir)
    ok = (out_dir / "REVIEWED").exists()
    return _item("Tinjauan orang kedua (berkas REVIEWED)", LULUS if ok else MENUNGGU,
                 "ada" if ok else f"setelah meninjau hasil: approve_review('{out_dir.name.upper()}', 'NAMA') (RULES §3)")


def _flag_item(out_dir, name: str, nama: str, how: str) -> dict:
    ok = (Path(out_dir) / name).exists()
    return _item(nama, LULUS if ok else MENUNGGU, "ada" if ok else how)


def _decision_item(d: str) -> dict:
    with open(DEC_PATH, "r", encoding="utf-8") as f:
        dec = yaml.safe_load(f)[d]
    st = dec["status"]
    status = {"diputuskan": LULUS, "ditunda": LULUS, "menunggu_konfirmasi": MENUNGGU}.get(st, GAGAL)
    return _item(f"{d} {st}", status, str(dec.get("isi", "")))


def _finish(cp: str, out_dir, items: list, files: list, extra=None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    blocking = [i for i in items if i["status"] in (GAGAL, MENUNGGU)]
    rep = {"checkpoint": cp, "lulus": not blocking, "items": items, "penghambat": [i["item"] for i in blocking]}
    atomic_write_json(out_dir / f"{cp.lower()}_check.json", rep)
    for i in items:
        print(f"[{i['status']:>15}] {i['item']} -- {i['detail']}")
    if rep["lulus"]:
        write_manifest(out_dir, [out_dir / f"{cp.lower()}_check.json", *[f for f in files if Path(f).exists()]],
                       extra={"checkpoint": cp, "git_commit": GIT_COMMIT, **(extra or {})})
        print(f"\n{cp} LULUS. manifest.json ditulis. Tag git '{cp.lower()}' dapat dibuat (setelah tinjauan orang kedua).")
    else:
        print(f"\n{cp} BELUM LULUS. Penghambat: {rep['penghambat']}\nmanifest.json TIDAK ditulis.")
    return rep


def log_row(who: str, what: str, hasil: str = "disetujui", keputusan: str = "via notebook") -> None:
    line = f"| {dt.date.today().isoformat()} | {who} | {what} | {hasil} | {keputusan} |\n"
    if LOG_PATH.exists():
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line)


def approve_flag(cp: str, name: str, by: str, what: str) -> None:
    d = P(load_merged(BASE_PATH), cp.lower())
    d.mkdir(parents=True, exist_ok=True)
    (d / name).write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, f"{what} ({cp})")
    print("ditulis", d / name)


def approve_review(cp: str, by: str) -> None:
    approve_flag(cp, "REVIEWED", by, "Tinjauan orang kedua")


def approve_decisions(ids: list, by: str) -> None:
    text = DEC_PATH.read_text(encoding="utf-8")
    for d in ids:
        pat = re.compile(rf"(^{d}:\n  status: )menunggu_konfirmasi", re.M)
        if not pat.search(text):
            print(f"{d}: tidak berstatus menunggu_konfirmasi (dilewati)")
            continue
        text = pat.sub(rf"\1diputuskan\n  dikonfirmasi_oleh: {by}\n  tanggal_konfirmasi: {dt.date.today().isoformat()}", text)
        log_row(by, f"Konfirmasi keputusan {d}")
        print(f"{d}: diputuskan oleh {by}")
    DEC_PATH.write_text(text, encoding="utf-8")


def approve_roi(by: str) -> None:
    t = BASE_PATH.read_text(encoding="utf-8")
    t2 = re.sub(r"(  roi:.*?\n    locked: )false", r"\1true", t, count=1, flags=re.S)
    if t2 == t and "locked: true" not in t:
        raise SystemExit("preprocess.roi.locked tidak ditemukan di base.yaml")
    BASE_PATH.write_text(t2, encoding="utf-8")
    d = P(load_merged(BASE_PATH), "cp2")
    d.mkdir(parents=True, exist_ok=True)
    (d / "ROI_APPROVED").write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, "Parameter Frangi dikunci setelah pemeriksaan visual 50 citra (FR-03)")
    print("preprocess.roi.locked=true; ditulis", d / "ROI_APPROVED")

## 3. Kode audit (dari `audit.py` Versi 1)

In [ ]:
# ---------------------------------------------------------------- audit data (dari audit.py Versi 1, FR-01)
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png"}
CLASSES = (("Benign", 0), ("Malignant", 1))


def _size(path: Path):
    with Image.open(path) as im:
        return im.size


def scan_dataset(root) -> pd.DataFrame:
    # Satu baris per citra: Benign|Malignant / <histopatologi> / <PatientXXX> / berkas.
    root = Path(root)
    files = []
    for class_name, label in CLASSES:
        cdir = root / class_name
        for histo in sorted(p for p in cdir.iterdir() if p.is_dir()):
            for pdir in sorted(p for p in histo.iterdir() if p.is_dir()):
                for f in sorted(pdir.iterdir()):
                    if f.suffix.lower() in IMG_EXT:
                        files.append((f, class_name, label, histo.name, pdir.name))
    with ThreadPoolExecutor(8) as ex:
        sizes = list(ex.map(lambda t: _size(t[0]), files))
    rows = [dict(rel_path=str(f.relative_to(root)).replace("\\", "/"), label=label, class_name=cn,
                 histo=h, patient_id=pid, width=w, height=hh, bytes=f.stat().st_size)
            for (f, cn, label, h, pid), (w, hh) in zip(files, sizes)]
    return pd.DataFrame(rows)


def assign_uid(df: pd.DataFrame):
    labels_per_id = df.groupby("patient_id")["label"].nunique()
    collisions = sorted(labels_per_id[labels_per_id > 1].index)
    df = df.copy()
    df["patient_uid"] = [f"{c}_{p}" if p in collisions else p for c, p in zip(df.class_name, df.patient_id)]
    return df, collisions


def patients_table(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["resolution"] = df.width.astype(str) + "x" + df.height.astype(str)
    g = df.groupby("patient_uid")
    t = pd.DataFrame({
        "label": g["label"].first(), "class_name": g["class_name"].first(), "histo": g["histo"].first(),
        "n_histo_folders": g["histo"].nunique(), "n_images": g.size(), "n_resolutions": g["resolution"].nunique(),
        "resolution": g["resolution"].agg(lambda s: s.value_counts().index[0]),
    })
    t["label_consistent"] = g["label"].nunique() == 1
    return t.reset_index()


def reference_comparison(pt: pd.DataFrame, img: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    loc = {("patients", "benign"): int((pt.label == 0).sum()), ("patients", "malignant"): int((pt.label == 1).sum()),
           ("images", "benign"): int((img.label == 0).sum()), ("images", "malignant"): int((img.label == 1).sum())}
    rows = []
    for (unit, cls), v in loc.items():
        r = {"unit": unit, "kelas": cls, "lokal": v}
        for name in ("reference_tabel_3_2", "reference_kiani"):
            ref = cfg["audit"][name][unit][cls]
            r[name] = ref
            r[f"selisih_{name}"] = v - ref
        rows.append(r)
    rows.append({"unit": "images", "kelas": "total", "lokal": len(img),
                 "reference_tabel_3_2": sum(cfg["audit"]["reference_tabel_3_2"]["images"].values()),
                 "reference_kiani": sum(cfg["audit"]["reference_kiani"]["images"].values())})
    rows[-1]["selisih_reference_tabel_3_2"] = rows[-1]["lokal"] - rows[-1]["reference_tabel_3_2"]
    rows[-1]["selisih_reference_kiani"] = rows[-1]["lokal"] - rows[-1]["reference_kiani"]
    return pd.DataFrame(rows)


def s7_check(pt: pd.DataFrame, min_per_class: int) -> dict:
    # Syarat S7 (PRD §6): latih pada resolusi mayoritas, uji pada gabungan resolusi lain; minimal N pasien per kelas.
    single = pt[pt.n_resolutions == 1]
    per_res = []
    for res, g in single.groupby("resolution"):
        b, m = int((g.label == 0).sum()), int((g.label == 1).sum())
        per_res.append({"resolution": res, "benign_patients": b, "malignant_patients": m, "images": int(g.n_images.sum()),
                        "malignant_fraction": round(m / max(b + m, 1), 3), "memenuhi_syarat": b >= min_per_class and m >= min_per_class})
    per_res.sort(key=lambda r: -(r["benign_patients"] + r["malignant_patients"]))
    out = {"per_resolusi": per_res, "pasien_multi_resolusi": int((pt.n_resolutions > 1).sum())}
    if per_res:
        major = per_res[0]["resolution"]
        minor = single[single.resolution != major]
        b, m = int((minor.label == 0).sum()), int((minor.label == 1).sum())
        out["mayoritas"] = per_res[0]
        out["minor_gabungan"] = {"benign_patients": b, "malignant_patients": m,
                                 "memenuhi_syarat": b >= min_per_class and m >= min_per_class}
        out["s7_dapat_dijalankan"] = bool(per_res[0]["memenuhi_syarat"] and out["minor_gabungan"]["memenuhi_syarat"]
                                          and out["pasien_multi_resolusi"] == 0)
    return out


def _pid_num(s):
    m = re.findall(r"\d+", str(s))
    return int(m[-1]) if m else None


def cross_check_xlsx(xlsx_path, pt: pd.DataFrame) -> dict:
    try:
        sheets = pd.read_excel(xlsx_path, sheet_name=None)
    except Exception as e:  # noqa: BLE001
        return {"status": "tidak_dapat_dibaca", "detail": repr(e)}
    out = {"status": "dibaca", "sheet": {k: list(map(str, v.columns)) for k, v in sheets.items()}}
    df = max(sheets.values(), key=len)
    cols = {str(c).lower(): c for c in df.columns}
    pcol = next((c for k, c in cols.items() if "patient" in k or k in ("id", "pid", "no")), None)
    lcol = next((c for k, c in cols.items() if any(w in k for w in ("class", "label", "malig", "benign", "diagn", "type", "categ"))), None)
    out["kolom_pasien"], out["kolom_label"] = (str(pcol) if pcol is not None else None), (str(lcol) if lcol is not None else None)
    if pcol is None:
        out["detail"] = "kolom ID pasien tidak terdeteksi; pemeriksaan silang dilewati"
        return out
    x_ids = {_pid_num(v) for v in df[pcol].dropna()}
    f_nums = [_pid_num(v) for v in pt.patient_uid]
    f_ids = set(f_nums)
    ganda = sorted({n for n in f_nums if n is not None and f_nums.count(n) > 1})
    out["nomor_numerik_ganda_di_folder"] = ganda      # dua folder pasien dengan angka sama: dikecualikan dari pencocokan label/jumlah/resolusi
    out.update(pasien_xlsx=len(x_ids), pasien_folder=len(f_ids),
               hanya_di_xlsx=sorted(i for i in x_ids - f_ids if i is not None),
               hanya_di_folder=sorted(i for i in f_ids - x_ids if i is not None))
    if lcol is not None:
        def to_label(v):
            s = str(v).lower()
            return 1 if ("malig" in s or s.strip() in ("1", "ganas")) else (0 if ("benign" in s or "jinak" in s or s.strip() == "0") else None)
        xl = {_pid_num(a): to_label(b) for a, b in zip(df[pcol], df[lcol]) if _pid_num(a) is not None}
        fl = {_pid_num(a): b for a, b in zip(pt.patient_uid, pt.label) if _pid_num(a) not in ganda}
        out["label_beda_xlsx_vs_folder"] = sorted(i for i in xl if i in fl and xl[i] is not None and xl[i] != fl[i])
        out["label_xlsx_tidak_terbaca"] = sum(v is None for v in xl.values())
    else:
        out["detail"] = "kolom label tidak terdeteksi; label tidak dibandingkan"
    ncol = next((c for k, c in cols.items() if "number" in k and "image" in k), None)
    scol = next((c for k, c in cols.items() if "size" in k), None)
    fp = {_pid_num(a): r for a, r in zip(pt.patient_uid, pt.itertuples()) if _pid_num(a) not in ganda}
    if ncol is not None:
        xn = {_pid_num(a): b for a, b in zip(df[pcol], df[ncol]) if _pid_num(a) is not None and pd.notna(b)}
        out["jumlah_citra_beda"] = sorted([{"pasien": i, "xlsx": int(xn[i]), "folder": int(fp[i].n_images)}
                                           for i in xn if i in fp and int(xn[i]) != int(fp[i].n_images)], key=lambda d: d["pasien"])
        out["jumlah_citra_total_xlsx"] = int(sum(xn.values()))
    if scol is not None:
        def norm(v):
            m = re.findall(r"\d+", str(v))
            return f"{m[0]}x{m[1]}" if len(m) >= 2 else None
        xs = {_pid_num(a): norm(b) for a, b in zip(df[pcol], df[scol]) if _pid_num(a) is not None and pd.notna(b)}
        out["resolusi_beda"] = sorted([{"pasien": i, "xlsx": xs[i], "folder": fp[i].resolution}
                                       for i in xs if i in fp and xs[i] != fp[i].resolution], key=lambda d: d["pasien"])
    return out


def run_audit(cfg: dict, out_dir=None) -> dict:
    out_dir = Path(out_dir or P(cfg, "cp1"))
    out_dir.mkdir(parents=True, exist_ok=True)
    root = Path(cfg["paths"]["dataset"])
    img = scan_dataset(root)
    img, collisions = assign_uid(img)
    pt = patients_table(img)
    ref = reference_comparison(pt, img, cfg)
    s7 = s7_check(pt, cfg["audit"]["s7_min_patients_per_class"])
    xlsx = root / "Patients_List_Updated_Final.xlsx"
    xc = cross_check_xlsx(xlsx, pt) if xlsx.exists() else {"status": "tidak_ada"}
    atomic_write_bytes(out_dir / "audit.csv", pt.to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(out_dir / "audit_images.csv", img.to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(out_dir / "audit_reference.csv", ref.to_csv(index=False).encode("utf-8"))
    summary = {"n_images": len(img), "n_patients": len(pt), "id_bertabrakan_antar_kelas": collisions,
               "pasien_label_tidak_konsisten": int((~pt.label_consistent).sum()),
               "pasien_di_lebih_dari_satu_folder_histo": int((pt.n_histo_folders > 1).sum()),
               "perbandingan_referensi": ref.to_dict(orient="records"), "s7": s7, "xlsx": xc,
               "resolusi_per_citra": img.assign(r=img.width.astype(str) + "x" + img.height.astype(str)).r.value_counts().to_dict()}
    atomic_write_json(out_dir / "audit_summary.json", summary)
    return summary

## 4. Jalankan audit (dilewati bila `audit.csv` sudah ada)

In [ ]:
if (P(cfg, 'cp1', 'audit.csv')).exists():
    print('[dilewati] audit.csv sudah ada; hapus outputs/cp1 untuk mengulang')
else:
    run_audit(cfg)
s = json.loads(P(cfg, 'cp1', 'audit_summary.json').read_text(encoding='utf-8'))
print(f"citra={s['n_images']} pasien={s['n_patients']}")

## 5. Perbandingan dengan Tabel 3.2 dan repo Kiani

In [ ]:
display(pd.DataFrame(s['perbandingan_referensi']))
print('ID pasien bertabrakan antar kelas:', s['id_bertabrakan_antar_kelas'])
print('Pasien label tidak konsisten:', s['pasien_label_tidak_konsisten'])
print('Pasien di >1 folder histopatologi:', s['pasien_di_lebih_dari_satu_folder_histo'])

## 6. Silang-periksa dengan `Patients_List_Updated_Final.xlsx`

In [ ]:
for k, v in s['xlsx'].items():
    print(k, ':', (v if not isinstance(v, list) or len(v) <= 15 else v[:15] + ['...']))

## 7. Resolusi dan syarat S7 (dihitung ulang dari gambar lokal)

In [ ]:
display(pd.DataFrame(s['s7']['per_resolusi']))
print('mayoritas:', s['s7'].get('mayoritas'))
print('minor gabungan:', s['s7'].get('minor_gabungan'))
print('pasien dengan >1 resolusi:', s['s7']['pasien_multi_resolusi'])
print('S7 dapat dijalankan:', s['s7'].get('s7_dapat_dijalankan'))

## 8. Pemeriksaan checkpoint CP1 dan `manifest.json` (sel terakhir)

Selisih terhadap Tabel 3.2 (bila ada) berstatus `MENUNGGU` sampai dijelaskan di `LOG.md`; tinjauan orang kedua lewat sel di bawah.

In [ ]:
# Isi nama peninjau KEDUA setelah benar-benar meninjau hasil di atas (RULES §3), lalu jalankan sel ini.
REVIEWER = None      # contoh: 'Ahmad'
if REVIEWER:
    approve_review('CP1', REVIEWER)

In [ ]:
def finalize_cp1(cfg: dict) -> dict:
    out = P(cfg, "cp1")
    items = []
    need = [out / n for n in ("audit.csv", "audit_images.csv", "audit_summary.json", "audit_reference.csv")]
    missing = [p.name for p in need if not p.exists()]
    if missing:
        items.append(_item("Keluaran audit lengkap", GAGAL, f"hilang: {missing}"))
        return _finish("CP1", out, items, [])
    s = json.loads((out / "audit_summary.json").read_text(encoding="utf-8"))
    items.append(_item("audit.csv dibuat dari gambar lokal", LULUS, f"{s['n_patients']} pasien, {s['n_images']} citra"))
    ref = pd.DataFrame(s["perbandingan_referensi"])
    diff = ref[ref.selisih_reference_tabel_3_2 != 0]
    if diff.empty:
        items.append(_item("Angka sama dengan Tabel 3.2", LULUS, "pasien dan citra per kelas identik"))
    else:
        items.append(_item("Angka sama dengan Tabel 3.2", MENUNGGU, "selisih ada, peneliti harus menjelaskan: " + "; ".join(
            f"{r.unit}/{r.kelas} lokal={r.lokal} vs {r.reference_tabel_3_2}" for r in diff.itertuples())))
    kd = ref[ref.selisih_reference_kiani != 0]
    items.append(_item("Selisih terhadap repo Kiani dilaporkan (informasi)", LULUS,
                       "; ".join(f"{r.unit}/{r.kelas}: {int(r.selisih_reference_kiani):+d}" for r in kd.itertuples()) or "tidak ada selisih"))
    col = s["id_bertabrakan_antar_kelas"]
    items.append(_item("ID pasien unik antar kelas", LULUS if not col else MENUNGGU,
                       "tidak ada tabrakan" if not col else f"ID di kedua kelas (diberi awalan kelas): {col[:10]}"))
    bad = s["pasien_label_tidak_konsisten"] + s["pasien_di_lebih_dari_satu_folder_histo"]
    items.append(_item("Label dan folder histopatologi konsisten per pasien", LULUS if bad == 0 else MENUNGGU,
                       f"label tidak konsisten={s['pasien_label_tidak_konsisten']}, pasien di >1 folder histo={s['pasien_di_lebih_dari_satu_folder_histo']}"))
    x = s["xlsx"]
    if x.get("status") != "dibaca" or x.get("kolom_pasien") is None:
        items.append(_item("Silang-periksa Patients_List_Updated_Final.xlsx", MENUNGGU, f"tidak dapat dilakukan: {x}"))
    else:
        probs = {"hanya_di_xlsx": len(x.get("hanya_di_xlsx", [])), "hanya_di_folder": len(x.get("hanya_di_folder", [])),
                 "label_beda": len(x.get("label_beda_xlsx_vs_folder", [])), "jumlah_citra_beda": len(x.get("jumlah_citra_beda", [])),
                 "resolusi_beda": len(x.get("resolusi_beda", []))}
        items.append(_item("Silang-periksa xlsx (pasien, label, jumlah citra, resolusi)", LULUS if not any(probs.values()) else MENUNGGU, str(probs)))
    s7 = s["s7"]
    items.append(_item("Syarat S7 dihitung ulang dari gambar lokal", LULUS,
                       f"dapat dijalankan={s7.get('s7_dapat_dijalankan')}; mayoritas={s7.get('mayoritas', {}).get('resolution')}; "
                       f"minor gabungan={s7.get('minor_gabungan')}; pasien multi-resolusi={s7.get('pasien_multi_resolusi')}"))
    items.append(_review_item(out))
    return _finish("CP1", out, items, need)


report = finalize_cp1(cfg)
assert report['checkpoint'] == 'CP1'